# DSSATLab walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdelrahmanAmr3/dssatlab/blob/master/notebook/dssatlab_walkthrough.ipynb)

A hands-on tour of `dssatlab`, using the latest code from GitHub (before the next PyPI release):

1. `connect()` finds DSSAT and `install()` builds it on Linux/Colab (sections 1-4).
2. `Simulation` runs one treatment of a FileX with **your own weather data** (sections 5-7).
3. The same `Simulation` with your own **soil** and **management** data (sections 8-9).

Run the cells top to bottom. The build in section 2 takes several minutes. On a machine that already has DSSAT (for example Windows), skip section 2 and run the rest.

## Setup

In [ ]:
%pip install -q "git+https://github.com/AbdelrahmanAmr3/dssatlab.git"

In [ ]:
import logging
import dssatlab as dl

# dssatlab logs how it finds DSSAT; show those messages here.
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
print("dssatlab", dl.__version__)

## 1. `connect()` before installing

On a fresh runtime DSSAT is not installed, so we expect `DSSATNotFoundError`. It lists what was checked and what to do next.
`interactive=False` raises the error instead of asking a question (normally `dl.connect()` would offer to install).

In [ ]:
try:
    print(dl.connect(interactive=False))
except dl.DSSATNotFoundError as e:
    print(e)

## 2. `install()`

Downloads the latest stable DSSAT release from GitHub and builds it with git, cmake and gfortran (already on Colab).

In [ ]:
import time

t0 = time.time()
dssat = dl.install()
print(dssat)
print(f"took {time.time() - t0:.0f}s")

## 3. `connect()` after installing

In [ ]:
# The build was remembered, so connect() now returns it straight away.
assert dl.connect(interactive=False) == dssat

## 4. `install()` again

The build is reused, so this should take a moment instead of minutes.

In [ ]:
t0 = time.time()
again = dl.install()
print(again, f"({time.time() - t0:.1f}s)")
assert again == dssat

## 5. Get a sample FileX

`Simulation` needs a FileX and the genotype files beside it. We download DSSAT's own Gainesville maize experiment (`UFGA8201.MZX`) from the official DSSAT repositories into a folder called `demo`. Its weather file is downloaded too, but only so we can turn it into a CSV in the next section; the `Simulation` does not use it.

In [ ]:
import urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/DSSAT/dssat-csm-os/develop/Data"
EXPERIMENTS = "https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop"
demo = Path("demo")
demo.mkdir(exist_ok=True)

files = {
    "UFGA8201.MZX": f"{EXPERIMENTS}/Maize/UFGA8201.MZX",
    "UFGA8201.WTH": f"{EXPERIMENTS}/Weather/UFGA8201.WTH",
    "MZCER048.CUL": f"{DATA}/Genotype/MZCER048.CUL",
    "MZCER048.ECO": f"{DATA}/Genotype/MZCER048.ECO",
    "MZCER048.SPE": f"{DATA}/Genotype/MZCER048.SPE",
}
for name, url in files.items():
    if not (demo / name).exists():
        urllib.request.urlretrieve(url, demo / name)
print(sorted(p.name for p in demo.iterdir()))

## 6. Your weather data as a CSV

`Simulation` reads daily weather from a CSV with fixed column names. `write_weather_template()` writes an example you can edit. Here we skip the editing: we convert the sample `.WTH` into that CSV layout, so you can see what real data looks like.

In [ ]:
import csv

dl.write_weather_template("weather_template.csv")
print(Path("weather_template.csv").read_text().splitlines()[:3], "...")

In [ ]:
# Turn the sample UFGA8201.WTH into a weather CSV (this is the part you would do with your own data).
from datetime import date, timedelta

lines = (demo / "UFGA8201.WTH").read_text().splitlines()
site = lines[next(i for i, l in enumerate(lines) if l.startswith("@ INSI")) + 1].split()
start = next(i for i, l in enumerate(lines) if l.startswith("@DATE")) + 1
# site line: station latitude longitude elevation tav amp refht wndht

rows = []
for line in lines[start:]:
    if not line.strip():
        continue
    d, srad, tmax, tmin, rain = line.split()[:5]
    day = date(1900 + int(d[:2]), 1, 1) + timedelta(days=int(d[2:]) - 1)
    rows.append({"station": site[0], "latitude": site[1], "longitude": site[2],
                 "elevation": site[3], "date": day.isoformat(), "srad": srad,
                 "tmax": tmax, "tmin": tmin, "rain": rain,
                 "tav": site[4], "amp": site[5], "refht": site[6], "wndht": site[7]})

with open("weather.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print(len(rows), "days:", rows[0]["date"], "to", rows[-1]["date"])

## 7. Check and run a `Simulation`

`Simulation(...)` only stores its inputs. `check()` reads the weather and FileX and returns a list of problems without writing anything; an empty list means it is ready. `run()` repeats the checks, copies the FileX and genotype files into a fresh `dssat_sim_...` folder, writes the weather file, and runs DSSAT there. Your originals are left untouched.

In [ ]:
sim = dl.Simulation(demo / "UFGA8201.MZX", treatment=1, weather="weather.csv")
problems = sim.check(verbose=True)
print("problems:", problems)

In [ ]:
result = sim.run()
print("exit status:", result.returncode)
print("run directory:", result.run_dir)
print([p.name for p in result.outputs])

### Look at the output

`Summary.OUT` has one fixed-width line per treatment run. `dssatlab` does not parse DSSAT output yet, so this small helper reads one column of it.

Sanity check: running this same FileX directly in DSSAT 4.8.5 (treatment 1, its own `.WTH` and soil) gives `HWAM` = 2293 kg/ha. You should get almost the same number here, because the CSV carries the same weather. Treatment 3 (the same crop with irrigation) gives 8207.

In [ ]:
def summary_value(res, column):
    """Read one column of the last line of Summary.OUT (fixed-width, right-aligned)."""
    out = next(p for p in res.outputs if p.name.lower() == "summary.out")
    lines = out.read_text().splitlines()
    header = next(l for l in reversed(lines) if l.startswith("@"))
    end = header.index(" " + column + " ") + len(column) + 1
    return lines[-1][:end].split()[-1]

print("Harvested grain yield (HWAM, kg/ha):", summary_value(result, "HWAM"))
print("Maturity date (MDAT, YYYYDDD):     ", summary_value(result, "MDAT"))

### What a mistake looks like

Here the weather station code does not match the FileX's `WSTA` (`UFGA`). `check()` reports it, and `run()` refuses to start DSSAT.

In [ ]:
bad = [dict(row, station="XXXX") for row in rows]
try:
    dl.Simulation(demo / "UFGA8201.MZX", treatment=1, weather=bad).run()
except dl.DSSATCheckError as error:
    for problem in error.problems:
        print("-", problem)

## 8. Your own soil data

Pass `soil=` and `dssatlab` writes a `SOIL.SOL` from it instead of using a soil file. The soil is one profile, one row per layer, and its `soil_id` must equal the FileX's `ID_SOIL` (`IBMZ910014` for treatment 1).

`write_soil_template()` writes an editable example CSV. Here we type in the real Gainesville Millhopper fine sand profile (8 layers to 180 cm) that DSSAT itself uses for this experiment, so the yield should stay close to section 7.

In [ ]:
dl.write_soil_template("soil_template.csv")   # an example you can edit; not used below

profile = {"soil_id": "IBMZ910014", "salb": 0.18, "slro": 60, "sldr": 0.65, "slpf": 0.92,
           "slnf": 1.0, "slu1": 2.0}
# slb (bottom depth, cm), slll, sdul, ssat, srgf, sbdm, sloc
layers = [(5, .026, .096, .23, 1.0, 1.30, 2.00), (15, .025, .086, .23, 1.0, 1.30, 1.00),
          (30, .025, .086, .23, 0.7, 1.40, 1.00), (60, .025, .086, .23, 0.3, 1.40, 0.50),
          (90, .028, .090, .23, 0.05, 1.45, 0.10), (120, .028, .090, .23, 0.03, 1.45, 0.10),
          (150, .029, .130, .23, 0.002, 1.45, 0.04), (180, .070, .258, .36, 0.0, 1.20, 0.24)]
soil = [dict(profile, slb=b, slll=ll, sdul=dul, ssat=sat, srgf=rgf, sbdm=bd, sloc=oc)
        for b, ll, dul, sat, rgf, bd, oc in layers]

sim = dl.Simulation(demo / "UFGA8201.MZX", treatment=1, weather="weather.csv", soil=soil)
print("problems:", sim.check())
result_soil = sim.run()
print("HWAM with own soil:", summary_value(result_soil, "HWAM"), "kg/ha")

## 9. Your own management data

Pass `management=` to replace planting, irrigation or fertilizer for the selected treatment. It can be a YAML file (`write_management_template()` writes a documented one, needs `pip install pyyaml`) or a plain dict as below. Dates are quoted `"YYYY-MM-DD"` strings.

Treatment 1 is rainfed. Here we give it the 12-event irrigation schedule that the FileX itself uses for its irrigated treatment 3. The yield should jump to roughly what treatment 3 gives (8207 kg/ha in DSSAT 4.8.5).

In [ ]:
# Irrigation days (1982 day of year) and amounts (mm) from the FileX's irrigated level.
schedule = [(63, 13), (77, 10), (94, 10), (107, 13), (111, 18), (122, 25),
            (126, 25), (129, 13), (132, 15), (134, 19), (137, 20), (141, 20)]
management = {"treatments": {1: {"irrigation": [
    {"date": (date(1982, 1, 1) + timedelta(days=doy - 1)).isoformat(), "amount": float(mm),
     "method": "IR001"} for doy, mm in schedule
]}}}

sim = dl.Simulation(demo / "UFGA8201.MZX", treatment=1, weather="weather.csv",
                    management=management)
print("problems:", sim.check())   # management given, so the report is printed
result_mgmt = sim.run()
print("HWAM with irrigation:", summary_value(result_mgmt, "HWAM"), "kg/ha")

## 10. Compare

Own soil = same soil as the FileX, so the yield stays about the same. Added irrigation makes it much higher.

In [ ]:
for label, res in [("FileX defaults", result), ("own soil", result_soil), ("irrigation added", result_mgmt)]:
    print(f"{label:18s} HWAM = {summary_value(res, 'HWAM')} kg/ha")